# Hackathon #2 - Tweets Sentiment Analysis + RAG + Responses Generator

## 1. Define the Goal & Dataset

In [ ]:
! pip install datasets transformers -q

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "training.1600000.processed.noemoticon.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "kazanova/sentiment140",
  file_path,
  pandas_kwargs={"encoding": "ISO-8859-1", "header": None, "names": ["sentiment", "id", "date", "query", "user", "text"]}
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

Using Colab cache for faster access to the 'sentiment140' dataset.
First 5 records:    sentiment          id                          date     query  \
0          0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1          0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2          0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3          0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4          0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                               text  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  


In [ ]:
# Keeping only 10000 random items from df
df = df.sample(n=10000, random_state=42)

In [ ]:
# Only keep the "text" and "sentiment" columns
df = df[["text", "sentiment"]]

In [ ]:
# Map the sentiment labels 0 and 4 to 0 and 1
df['sentiment'] = df['sentiment'].replace({0: 0, 4: 1})

# Check the new distribution of sentiment labels
print("New distribution of sentiment labels:")
print(df['sentiment'].value_counts())

New distribution of sentiment labels:
sentiment
0    5004
1    4996
Name: count, dtype: int64


## 2. Choose a Base Model & Load Data

In [ ]:
BASE_MODEL = "distilbert/distilbert-base-uncased"

### Then, write code to load and tokenize your dataset with Hugging Face’s datasets and transformers.

In [ ]:
# create a training split and test split (0.1)
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

# Tokenize every value from the "text" columns
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

tokenized_dataset = []

train_df = map(lambda x: tokenizer(x, padding="max_length", truncation=True), train_df["text"])
test_df = map(lambda x: tokenizer(x, padding="max_length", truncation=True), test_df["text"])

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 3. Implement Parameter‑Efficient Fine‑Tuning (LoRA)

### First, integrate the **peft** or optimum library into your training script.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import torch

### Then, configure a LoRA adapter for your model, freezing base weights and only training low‑rank matrices.

#### Classification Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=2,  # Binary classification (Positive vs Negative Sentiment)
    )

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
    # Freeze base model weights first (optional but typical)
for name, param in model.named_parameters():
        param.requires_grad = False

In [ ]:
# Printing the layer names
for name, module in model.named_modules():
    print(name)


distilbert
distilbert.embeddings
distilbert.embeddings.word_embeddings
distilbert.embeddings.position_embeddings
distilbert.embeddings.LayerNorm
distilbert.embeddings.dropout
distilbert.transformer
distilbert.transformer.layer
distilbert.transformer.layer.0
distilbert.transformer.layer.0.attention
distilbert.transformer.layer.0.attention.dropout
distilbert.transformer.layer.0.attention.q_lin
distilbert.transformer.layer.0.attention.k_lin
distilbert.transformer.layer.0.attention.v_lin
distilbert.transformer.layer.0.attention.out_lin
distilbert.transformer.layer.0.sa_layer_norm
distilbert.transformer.layer.0.ffn
distilbert.transformer.layer.0.ffn.dropout
distilbert.transformer.layer.0.ffn.lin1
distilbert.transformer.layer.0.ffn.lin2
distilbert.transformer.layer.0.ffn.activation
distilbert.transformer.layer.0.output_layer_norm
distilbert.transformer.layer.1
distilbert.transformer.layer.1.attention
distilbert.transformer.layer.1.attention.dropout
distilbert.transformer.layer.1.attention.q

#### LoRA Adapter

In [ ]:
lora_config = LoraConfig(
  task_type=TaskType.SEQ_CLS,   # sequence classification
  inference_mode=False,
  r=8,                         # rank
  lora_alpha=32,
  lora_dropout=0.1,
  target_modules=["q_lin", "k_lin", "v_lin"],  # attempt to catch common module names
  bias="none",
)


In [ ]:
    # Create LoRA model wrapper that will add trainable low-rank matrices
model = get_peft_model(model, lora_config)

In [ ]:
    # After get_peft_model, PEFT parameters are trainable. Log number of trainable params:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable} / {total} ({100*trainable/total:.3f}%)")


Trainable params: 813314 / 67768324 (1.200%)


## 4. Train & Validate the Classifier

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary" if args.num_labels == 2 else "macro")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


In [ ]:

    # 7) Training arguments
    training_args = TrainingArguments(
        output_dir="my-model",
        per_device_train_batch_size="32",
        per_device_eval_batch_size="32",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=1e-4,
        weight_decay=0.01,
        num_train_epochs=5,
        fp16="fp16"and torch.cuda.is_available(),
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        gradient_accumulation_steps=1,
        save_total_limit=3,
        dataloader_num_workers=4,
        report_to="none",  # disable wandb/sagemaker by default
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_df,
        eval_dataset=test_df,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # 8) Train
    trainer.train()

    # 9) Evaluate (on validation)
    metrics = trainer.evaluate()
    print("Validation metrics:", metrics)

/tmp/ipython-input-3709342868.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


ValueError: The train_dataset does not implement __len__, max_steps has to be specified. The number of steps needs to be known in advance for the learning rate scheduler.